# Experiments on the Congressinal Dataset

#### Importing Relveant Libraries

In [1]:
import os
import json
import pandas as pd
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statistics
from matplotlib.colors import ListedColormap
from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA

## Loading the Dataset

In [3]:
A_df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/Answers.pkl')
Q_df = pd.read_pickle('/mnt/fstore/DataFiles/PickledFiles/Questions.pkl')

## Some Relevant Data

In [2]:
feature_names = [
                "ttr",
                "avg_wordlen",
                "word_count",
                "flesch_kincaid_grade_level",
                "smog_index",
                "coleman_liau_index",
                "lix",
                "bias_words",
                "assertatives",
                "factives",
                "hedges",
                "implicatives",
                "report_verbs",
                "positive_opinion_words",
                "negative_opinion_words",
                "vadneg",
                "vadneu",
                "vadpos",
                "wneg",
                "wpos",
                "wneu",
                "sneg",
                "spos",
                "sneu",
            ]

In [ ]:
sessions = ['108','109','110','111','112','113','114','115','116','117']
commmittees = A_df['Committee'].unique()
A_party_scores_dict = {}
A_majority_scores_dict = {}
Q_party_scores_dict = {}
Q_majority_scores_dict = {}
gov_A_party_scores_dict = {}
gov_A_majority_scores_dict = {}
gov_Q_party_scores_dict = {}
gov_Q_majority_scores_dict = {}

## Grid Search

In [4]:
def search_grid(X,y, params, clf):
    dummy_clf = DummyClassifier(strategy="most_frequent")
    dummy_clf.fit(X, y)
    print('Dummy Clssifier:', dummy_clf.score(X, y))
    gs1 = GridSearchCV(estimator=clf, param_grid= params, n_jobs=100, cv= 5, return_train_score=True)
    gs1.fit(X,y)
    bestParams = gs1.best_params_
    print(bestParams)
    print(gs1.best_score_)
    return pd.DataFrame(gs1.cv_results_), gs1.best_score_, gs1.best_params_


# congress = '117'
# chamber = 'House'
# print(congress)

# X = Q_df.loc[(Q_df['Party']!='I') & (Q_df['Congress'] == congress) & (Q_df['Chamber'] == chamber)][q_features].to_numpy()
# y = Q_df.loc[(Q_df['Party']!='I') & (Q_df['Congress'] == congress) & (Q_df['Chamber'] == chamber)]['Party'].to_numpy()

# print('Support :', len(y))

# parameters = {
#     'n_estimators' : [int(x) for x in np.linspace(10, 100, num = 10)],
#     'criterion':('gini', 'entropy'), 
#     'max_depth':[int(x) for x in np.linspace(10, 500, num = 246)],
#     'min_samples_split' : [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5],
#     'n_jobs' : [100]
# }

# rfc = RandomForestClassifier()
# etc = ExtraTreesClassifier()

# gs1 = GridSearchCV(estimator= etc, param_grid= parameters, n_jobs=100, cv=5, return_train_score=True)
# gs1.fit(X,y)
# bestParams = gs1.best_params_
# print(bestParams)
# print(gs1.best_score_)
# party_df = pd.DataFrame(gs1.cv_results_)


## Classification by Dataset

In [5]:
A_X = A_df[feature_names]
A_y1 = A_df['Party']
A_y2 = A_df['Majority']

Q_X = Q_df[feature_names]
Q_y1 = Q_df['Party']
Q_y2 = Q_df['Majority']

In [6]:
params = {
    'n_estimators' : [int(x) for x in np.linspace(10, 100, num = 10)],
    'criterion':('gini', 'entropy'), 
    'max_depth':[int(x) for x in np.linspace(10, 500, num = 246)],
    'min_samples_split' : [0.0001, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5],
    'n_jobs' : [100]
}

In [7]:
clf = RandomForestClassifier()

In [8]:
result = search_grid(A_X, A_y1, params=params, clf = clf)

Dummy Clssifier: 0.5122515882893656


## Classification by Session only

In [ ]:
for congress in sessions:
    A_X = A_df[A_df['Congress'] == congress][feature_names]
    A_y1 = A_df[A_df['Congress'] == congress]['Party']
    A_y2 = A_df[A_df['Congress'] == congress]['Majority']

    Q_X = Q_df[Q_df['Congress'] == congress][feature_names]
    Q_y1 = Q_df[Q_df['Congress'] == congress]['Party']
    Q_y2 = Q_df[Q_df['Congress'] == congress]['Majority']

    result = search_grid(A_X, A_y1, params=params, clf = clf)
    result = search_grid(A_X, A_y2, params=params, clf = clf)
    result = search_grid(Q_X, A_y1, params=params, clf = clf)
    result = search_grid(Q_X, Q_y1, params=params, clf = clf)

## Classification by Committee only

In [ ]:
for committee in commmittees:
    A_X = A_df[A_df['Committee'] == committee][feature_names]
    A_y1 = A_df[A_df['Committee'] == committee]['Party']
    A_y2 = A_df[A_df['Committee'] == committee]['Majority']

    Q_X = Q_df[Q_df['Committee'] == committee][feature_names]
    Q_y1 = Q_df[Q_df['Committee'] == committee]['Party']
    Q_y2 = Q_df[Q_df['Committee'] == committee]['Majority']

## Classification by Sessino and Committee

In [ ]:
for congress in sessions:
    for committee in commmittees:
        A_X = A_df[(A_df['Congress'] == congress) & (A_df['Committee'] == committee)][feature_names]
        A_y1 = A_df[(A_df['Congress'] == congress) & (A_df['Committee'] == committee)]['Party']
        A_y2 = A_df[(A_df['Congress'] == congress) & (A_df['Committee'] == committee)]['Majority']

        Q_X = Q_df[(Q_df['Congress'] == congress) & (Q_df['Committee'] == committee)][feature_names]
        Q_y1 = Q_df[(Q_df['Congress'] == congress) & (Q_df['Committee'] == committee)]['Party']
        Q_y2 = Q_df[(Q_df['Congress'] == congress) & (Q_df['Committee'] == committee)]['Majority']